Data Refinement (DR)
-

# Imports

In [1]:
import pandas as pd
import numpy as np
import zipfile
import os
from google.colab import drive

# Data AB

## Load Data

In [2]:
drive.mount('/content/drive')
zip_path = "/content/drive/Shared drives/Gestió de Projectes/Projecte/data/consum_anomalies_facturacio_complet_anonymized.zip"
extract_folder = "/content/drive/Shared drives/Gestió de Projectes/Projecte/data/Consum anomalies facturacio complet_anonymized"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_folder)

parquet_files = [f for f in os.listdir(extract_folder) if f.endswith('.parquet')]
if len(parquet_files) == 0:
    raise FileNotFoundError("No .parquet file found in the ZIP!")
parquet_path = os.path.join(extract_folder, parquet_files[0])

df_ab = pd.read_parquet(parquet_path)
print(f"Data loaded from {parquet_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data loaded from /content/drive/Shared drives/Gestió de Projectes/Projecte/data/Consum anomalies facturacio complet_anonymized/Consum anomalies facturacio complet_anonymized.parquet


## Reduce Dimensions
- We will remove the dimesnions POLISSA_SUBM, NUMEROSERIECONTADOR, and FECHA_HORA as they are identificators with no relevance to our conclusion.

In [3]:
cols_to_drop = ['POLISSA_SUBM', 'NUMEROSERIECONTADOR', 'FECHA_HORA']

df_ab = df_ab.drop(columns=cols_to_drop)

print(f"Dropped columns: {', '.join(cols_to_drop)}")

Dropped columns: POLISSA_SUBM, NUMEROSERIECONTADOR, FECHA_HORA


## Data Filtering
- We will remove rows with date types prior to 2022

In [4]:
# We will remove rows with date types prior to 2022
date_cols = ['START_DATE', 'END_DATE']

df_ab = df_ab[(df_ab[date_cols] >= '2022-01-01').all(axis=1)]

print("Rows with date types prior to 2022 removed")

Rows with date types prior to 2022 removed


## One-Hot Encoding
- We will use one hot encoding for the dimension US_AIGUA_SUBM.

In [5]:
# One-hot encoding for US_AIGUA_SUBM
cols_to_encode = ['US_AIGUA_SUBM']

df_ab = pd.get_dummies(df_ab, columns=cols_to_encode, dtype=int)

## Extract Date Fetures
- We will extract the difference from START_DATE and END_DATE in days.
- We will also split the dimensions START_DATE and END_DATE for year, mont, day.

In [6]:
# Extract difference form START_DATE and END_DATE
df_ab['DATE_DIFF'] = (df_ab['END_DATE'] - df_ab['START_DATE']).dt.days

# Split START_DATE in year, monts, day
df_ab['START_MONTH'] = df_ab.START_DATE.dt.month
df_ab['START_DAY'] = df_ab.START_DATE.dt.day

df_ab = df_ab.drop(columns=['START_DATE'])

# Split START_DATE in year, monts, day
df_ab['END_MONTH'] = df_ab.END_DATE.dt.month
df_ab['END_DAY'] = df_ab.END_DATE.dt.day

df_ab = df_ab.drop(columns=['END_DATE'])

## Handling Null Values
- SECCIO_CENSAL: we will remove rows with null values in this categor as it is the link with other datasets.
- CONSUMO_REAL: for this numerical features, we will use the median which is robust to outliers. We don't want to remove this rows as the null values account for roughly 20% of the dataset.

In [7]:
# Drop nulls
cols_to_drop = ['SECCIO_CENSAL']

df_ab = df_ab.dropna(subset=cols_to_drop)

print(f"Dropped rows with null values in '{', '.join(cols_to_drop)}'")

# Fill nulls
cols_to_fill = ['CONSUMO_REAL']

for col in cols_to_fill:
    median_val = df_ab[col].median()
    df_ab[col] = df_ab[col].fillna(median_val)
    print(f"Filled NAs in '{col}' with median: {median_val}")

Dropped rows with null values in 'SECCIO_CENSAL'
Filled NAs in 'CONSUMO_REAL' with median: 0.0


In [8]:
df_ab.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20833964 entries, 0 to 21195969
Data columns (total 12 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   CODI_ANOMALIA             int64  
 1   SECCIO_CENSAL             object 
 2   CONSUMO_REAL              float64
 3   US_AIGUA_SUBM_COMERCIAL   int64  
 4   US_AIGUA_SUBM_COMUNITARI  int64  
 5   US_AIGUA_SUBM_DOMÈSTIC    int64  
 6   US_AIGUA_SUBM_GENERAL     int64  
 7   DATE_DIFF                 int64  
 8   START_MONTH               int32  
 9   START_DAY                 int32  
 10  END_MONTH                 int32  
 11  END_DAY                   int32  
dtypes: float64(1), int32(4), int64(6), object(1)
memory usage: 1.7+ GB


In [9]:
df_ab.head()

,CODI_ANOMALIA,SECCIO_CENSAL,CONSUMO_REAL,US_AIGUA_SUBM_COMERCIAL,US_AIGUA_SUBM_COMUNITARI,US_AIGUA_SUBM_DOMÈSTIC,US_AIGUA_SUBM_GENERAL,DATE_DIFF,START_MONTH,START_DAY,END_MONTH,END_DAY
0,163840,0805601006,0.0,0,0,1,0,59,7,8,9,5
1,163840,0805602004,0.0,0,0,1,0,60,1,26,3,27
2,163840,0801507024,0.0,0,0,1,0,59,5,7,7,5
3,163840,0820002005,0.0,0,0,1,0,61,1,11,3,13
4,2,0801906059,0.0,0,0,1,0,59,1,16,3,16


# DATASET PREUS DE L'AIGUA PER MUNICIPI

## Load Data

In [10]:
drive.mount('/content/drive')
csv_path = "/content/drive/Shared drives/Gestió de Projectes/Projecte/data/Preus_per_municipi/Preus_municipi_ca_clean.csv"

df_preu = pd.read_csv(csv_path, sep=',', encoding='utf-8')
print(f"Data loaded from {csv_path}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data loaded from /content/drive/Shared drives/Gestió de Projectes/Projecte/data/Preus_per_municipi/Preus_municipi_ca_clean.csv


## Handling Null Values

**Subministrament (€/m³)**

És la variable més important del dataset, perquè explica gairebé tot el cost total.
Té només 2 valors nuls, per tant, no els eliminarem, ja que podrien ser municipis útils per l’anàlisi.

Solució: imputar pel valor mitjà de la seva comarca (manté coherència territorial).

In [11]:
df_preu['Subministrament (€/m³)'] = df_preu.groupby('Comarca')['Subministrament (€/m³)'].transform(
    lambda x: x.fillna(x.mean())
)

**Clavegueram (€/m³)**

Té diversos 0 o NaN perquè alguns municipis no tenen servei de clavegueram integrat. Això no és un error, sinó informació vàlida.
Solució: substituir NaN per 0.

In [12]:
df_preu['Clavegueram (€/m³)'] = df_preu['Clavegueram (€/m³)'].fillna(0)

**Tarifes socials**
Aquesta columna indica si el municipi ofereix tarifes especials o bonificades. Més de la meitat són NaN, però el motiu és que no s’han declarat, no pas que “no n’hi hagi”. Eliminar-la faria perdre informació potencial.
Solució: convertir-la en columna binària:
- 1 → té tarifes socials
- 0 → no té o no consta

In [13]:
df_preu['Tarifes socials'] = df_preu['Tarifes socials'].apply(lambda x: 1 if pd.notna(x) else 0)

In [14]:
df_preu.isnull().sum()

,0
Idescat,0
Municipi,0
Comarca,0
Subministrament (€/m³),0
Cànon de l'aigua (€/m³),0
Clavegueram (€/m³),0
TOTAL (€/m³),0
Entitat gestora principal Subministrament,0
Gestió Subministrament,0
Gestió Clavegueram,0


## Reduce Dimensions
- We will remove the dimesnions POLISSA_SUBM and NUMEROSERIECONTADOR as they are identificators with no relevance to our conclusion.
- We will also remove the dimension Cànon de l'aigua (€/m³) because we encountered that all the municipalities which from we have data have the same value. So adding this value will give no information.

In [15]:
cols_keep = [
    "Idescat",
    "Subministrament (€/m³)",
    "Clavegueram (€/m³)",
    "TOTAL (€/m³)",
    "Tarifes socials"
]

df_preu = df_preu[cols_keep]

print(f"Dropped kept: {', '.join(cols_keep)}")

Dropped kept: Idescat, Subministrament (€/m³), Clavegueram (€/m³), TOTAL (€/m³), Tarifes socials


In [16]:
df_preu.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947 entries, 0 to 946
Data columns (total 5 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Idescat                 947 non-null    int64  
 1   Subministrament (€/m³)  947 non-null    float64
 2   Clavegueram (€/m³)      947 non-null    float64
 3   TOTAL (€/m³)            947 non-null    float64
 4   Tarifes socials         947 non-null    int64  
dtypes: float64(3), int64(2)
memory usage: 37.1 KB


In [17]:
df_preu.head()

,Idescat,Subministrament (€/m³),Clavegueram (€/m³),TOTAL (€/m³),Tarifes socials
0,250019,0.150,0.000,0.583,0
1,80018,0.742,0.000,1.396,0
2,250024,1.360,0.000,2.014,0
3,250030,0.843,0.072,1.569,1
4,80023,1.973,0.348,2.754,0


# Merged Data

## Merge Data
We will save the data in the refined_data folder as one CSV.

https://www.idescat.cat/emex/?id=082009

https://do.diba.cat/data/ct/municipi/detall/08200

In [18]:
# Crear un diccionari de mapatge (Exemple fictici: SECCIO5 -> IDESCAT)
mapa_idescat = {
    '08056': '080569',  # Castelldefels
    '08015': '080155',  # Badalona
    '08200': '082009',  # Sant Boi de Llobregat
    '08019': '080193',  # Barcelona
    '08245': '082457',  # Santa Coloma de Gramenet
    '08301': '083015',  # Viladecans
    '08101': '081017',  # L'Hospitalet de Llobregat
    '08125': '081252',  # Montcada i Reixac
    '08266': '082665',  # Cerdanyola del Vallès
    '08077': '080771',  # Esplugues de Llobregat
    '08073': '080734',  # Cornellà de Llobregat
    '08157': '081574',  # Pallejà
    '08089': '080898',  # Gavà
    '08126': '081265',  # Montgat
    '08221': '082212',  # Sant Just Desvern
    '08211': '082114',  # Sant Feliu de Llobregat
    '08205': '082055',  # Sant Cugat del Vallès
    '08270': '082704',  # Sitges
    '08282': '082824',  # Tiana
    '08194': '081944',  # Sant Adrià de Besòs
    '08204': '082042',  # Sant Climent de Llobregat
    '08289': '082896',  # Torrelles de Llobregat
    '08180': '081803',  # Ripollet
    '08217': '082172',  # Sant Joan Despí
    '08020': '080207',  # Begues
}

# Crear una columna nova amb el codi IDESCAT
df_ab['Idescat'] = df_ab['SECCIO_CENSAL'].astype(str).str[:5].map(mapa_idescat)

df_preu['Idescat'] = df_preu['Idescat'].astype(str).str.zfill(6)

df_merged = df_ab.merge(df_preu, on='Idescat', how='left')

## Filter Data
- Remove dimensions SECCIO_CENSAL, Idescat

In [19]:
# Drop SECCIO_CENSAL and Idescat
cols_to_drop = ['SECCIO_CENSAL', 'Idescat']

df_merged = df_merged.drop(columns=cols_to_drop)

## Normalize Data
- We will normalize the dimensions CONSUMO_REAL, Subministrament (€/m³), Cànon de l'aigua (€/m³), Clavegueram (€/m³), and TOTAL (€/m³)

In [20]:
# Normalize dimensions CONSUMO_REAL, Subministrament (€/m³), Cànon de l'aigua (€/m³), Clavegueram (€/m³), and TOTAL (€/m³)
cols_to_normalize = ['CONSUMO_REAL', 'Subministrament (€/m³)', 'Clavegueram (€/m³)', 'TOTAL (€/m³)']

df_merged[cols_to_normalize] = df_merged[cols_to_normalize].apply(lambda x: (x - x.min()) / (x.max() - x.min()))

## Final Form

In [21]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20833964 entries, 0 to 20833963
Data columns (total 15 columns):
 #   Column                    Dtype  
---  ------                    -----  
 0   CODI_ANOMALIA             int64  
 1   CONSUMO_REAL              float64
 2   US_AIGUA_SUBM_COMERCIAL   int64  
 3   US_AIGUA_SUBM_COMUNITARI  int64  
 4   US_AIGUA_SUBM_DOMÈSTIC    int64  
 5   US_AIGUA_SUBM_GENERAL     int64  
 6   DATE_DIFF                 int64  
 7   START_MONTH               int32  
 8   START_DAY                 int32  
 9   END_MONTH                 int32  
 10  END_DAY                   int32  
 11  Subministrament (€/m³)    float64
 12  Clavegueram (€/m³)        float64
 13  TOTAL (€/m³)              float64
 14  Tarifes socials           int64  
dtypes: float64(4), int32(4), int64(7)
memory usage: 2.0 GB


In [22]:
df_merged.head()

,CODI_ANOMALIA,CONSUMO_REAL,US_AIGUA_SUBM_COMERCIAL,US_AIGUA_SUBM_COMUNITARI,US_AIGUA_SUBM_DOMÈSTIC,US_AIGUA_SUBM_GENERAL,DATE_DIFF,START_MONTH,START_DAY,END_MONTH,END_DAY,Subministrament (€/m³),Clavegueram (€/m³),TOTAL (€/m³),Tarifes socials
0,163840,0.030727,0,0,1,0,59,7,8,9,5,1.0,0.762402,0.954150,1
1,163840,0.030727,0,0,1,0,60,1,26,3,27,1.0,0.762402,0.954150,1
2,163840,0.030727,0,0,1,0,59,5,7,7,5,1.0,0.000000,0.723320,1
3,163840,0.030727,0,0,1,0,61,1,11,3,13,1.0,0.000000,0.723320,1
4,2,0.030727,0,0,1,0,59,1,16,3,16,1.0,0.733681,0.945455,1


In [23]:
df_merged.isnull().sum()

,0
CODI_ANOMALIA,0
CONSUMO_REAL,0
US_AIGUA_SUBM_COMERCIAL,0
US_AIGUA_SUBM_COMUNITARI,0
US_AIGUA_SUBM_DOMÈSTIC,0
US_AIGUA_SUBM_GENERAL,0
DATE_DIFF,0
START_MONTH,0
START_DAY,0
END_MONTH,0


# Save Data

In [24]:
# Define the output CSV path
csv_path = "/content/drive/Shared drives/Gestió de Projectes/Projecte/refined_data/final_dataset.csv"

# Save the DataFrame as CSV
df_merged.to_csv(csv_path, index=False, encoding='utf-8')

print(f"Final DataFrame saved to {csv_path}")

Final DataFrame saved to /content/drive/Shared drives/Gestió de Projectes/Projecte/refined_data/final_dataset.csv
